# Mobil Yüz Rekonstrüksiyon — Eğitim Notebook'u

**Mimari:** MLP Mapping Head + Depthwise Separable Upsample Decoder  
**Giriş:** 512-dim ArcFace embedding  
**Çıktı:** 128×128 RGB yüz  
**Platform:** Google Colab T4/A100 GPU  

---
### Adımlar
1. Kurulum
2. Google Drive bağlantısı
3. Proje kodunu Drive'a kopyala
4. Veri seti hazırlama
5. Ön işleme (yüz tespiti + embedding)
6. Eğitim
7. TFLite export
8. Sonuçları görselleştir

## 1. Kurulum

In [ ]:
# GPU kontrol
!nvidia-smi

# Temel paketler
!pip install -q \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -q \
    insightface \
    onnxruntime-gpu \
    facenet-pytorch \
    pytorch-msssim \
    torchmetrics \
    opencv-python-headless \
    onnx \
    tqdm \
    tensorboard

# ai-edge-torch: PyTorch → TFLite (opsiyonel ama önerilir)
!pip install -q ai-edge-torch

print('Kurulum tamamlandı.')

## 2. Google Drive Bağlantısı

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Drive'da proje klasörü oluştur
PROJECT_DIR = '/content/drive/MyDrive/face_recon'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Proje klasörü: {PROJECT_DIR}')

## 3. Proje Kodunu Yükle

In [ ]:
# SEÇENEK A: GitHub'dan çek (repo oluşturduktan sonra)
# !git clone https://github.com/KULLANICI/face-recon.git /content/face-recon
# %cd /content/face-recon

# SEÇENEK B: Drive'a yüklediğiniz zip'i aç
# !cp /content/drive/MyDrive/face_recon/proje.zip /content/
# !unzip -q /content/proje.zip -d /content/face-recon
# %cd /content/face-recon

# SEÇENEK C: Colab dosya yükleyicisi ile yükle
from google.colab import files
print('Proje zip dosyasını buraya sürükleyin veya aşağıdaki butona tıklayın:')
# uploaded = files.upload()

In [ ]:
# Proje dizinine gir — src klasörünü import edebilmek için
import sys
PROJECT_CODE = '/content/face-recon'  # proje klasörünüzü buraya yazın
sys.path.insert(0, PROJECT_CODE)
%cd {PROJECT_CODE}
print('Çalışma dizini:', os.getcwd())

## 4. Config Ayarları

In [ ]:
from src.config import Config, DataConfig, ModelConfig, LossConfig, TrainConfig, ExportConfig

cfg = Config(
    data=DataConfig(
        data_dir    = '/content/drive/MyDrive/face_data/raw',
        processed_dir = '/content/drive/MyDrive/face_data/processed',
        image_size  = 128,
        val_split   = 0.05,
        num_workers = 2,  # Colab'da 2 yeterli
    ),
    model=ModelConfig(
        embedding_dim    = 512,
        initial_spatial  = 4,
        initial_channels = 192,
        decoder_channels = (192, 128, 96, 64, 32),
    ),
    loss=LossConfig(
        phase1_epochs = 10,
        phase2_epochs = 50,
        phase3_epochs = 40,
        vgg_layer     = 'relu3_3',
        facenet_input_size = 160,
    ),
    train=TrainConfig(
        epochs           = 100,
        batch_size       = 64,
        learning_rate    = 1e-4,
        weight_decay     = 1e-4,
        warmup_epochs    = 5,
        save_dir         = '/content/drive/MyDrive/face_recon/checkpoints',
        save_every_epochs = 5,
        keep_last_n      = 3,
        patience         = 10,
        use_amp          = True,
        log_dir          = '/content/drive/MyDrive/face_recon/logs',
        log_every_steps  = 50,
    ),
    export=ExportConfig(
        export_dir  = '/content/drive/MyDrive/face_recon/export',
        model_name  = 'face_decoder',
        quantize_int8 = True,
    ),
)

print(f'Toplam epoch: {cfg.total_epochs()}')
print(f'Phase 1 (warm-up): epoch 0-{cfg.loss.phase1_epochs}')
print(f'Phase 2 (main):    epoch {cfg.loss.phase1_epochs}-{cfg.loss.phase1_epochs + cfg.loss.phase2_epochs}')
print(f'Phase 3 (fine-tune): epoch {cfg.loss.phase1_epochs + cfg.loss.phase2_epochs}+')


## 5. Veri Seti Hazırlama

Beklenen yapı:
```
face_data/raw/
    n000002/   ← kimlik klasörü
        0001_01.jpg
        0002_01.jpg
    n000003/
        ...
```

**Önerilen veri setleri:**
- **MS1MV3 (HuggingFace):** `FaceRecognition/ms1m-retinaface-t1`  
- **VGGFace2 (Kaggle):** `heartkilla/vggface2` 
- **CASIA-WebFace:** lisanslı kaynaklardan indir

In [ ]:
# SEÇENEK: VGGFace2'nin küçük bir alt kümesini Kaggle'dan indir
# Kaggle API gerektirir: kaggle.json dosyasını Drive'a yükleyin

# !pip install -q kaggle
# !mkdir -p ~/.kaggle
# !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d heartkilla/vggface2 -p /content/face_data/
# !unzip -q /content/face_data/vggface2.zip -d {cfg.data.data_dir}

# SEÇENEK: Test için 100 kimlik × 20 görüntü sahte veri oluştur
import os
import numpy as np
from PIL import Image

DEMO_MODE = True  # Gerçek veri ile çalışırken False yapın

if DEMO_MODE:
    print('Demo modu: sentetik veri oluşturuluyor...')
    raw_dir = cfg.data.data_dir
    os.makedirs(raw_dir, exist_ok=True)
    for i in range(50):  # 50 kimlik
        id_dir = os.path.join(raw_dir, f'identity_{i:04d}')
        os.makedirs(id_dir, exist_ok=True)
        for j in range(20):  # 20 görüntü
            img = Image.fromarray(np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8))
            img.save(os.path.join(id_dir, f'img_{j:04d}.jpg'))
    print(f'Sentetik veri oluşturuldu: {raw_dir}')
    print('NOT: Gerçek eğitim için VGGFace2/MS1MV3 kullanın!')

## 6. Ön İşleme: Yüz Tespiti + ArcFace Embedding

In [ ]:
from src.data.preprocess import preprocess_dataset

if DEMO_MODE:
    # Demo modunda sahte embeddingler oluştur (insightface gerektirmez)
    print('Demo: Sahte embeddingler oluşturuluyor...')
    import numpy as np
    from pathlib import Path
    from PIL import Image
    import shutil

    raw_path = Path(cfg.data.data_dir)
    out_path = Path(cfg.data.processed_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    manifest_lines = []
    for id_dir in sorted(raw_path.iterdir()):
        if not id_dir.is_dir():
            continue
        out_id = out_path / id_dir.name
        out_id.mkdir(exist_ok=True)

        for img_file in sorted(id_dir.glob('*.jpg')):
            out_img = out_id / img_file.name
            out_emb = out_id / (img_file.stem + '.npy')

            shutil.copy(img_file, out_img)
            # L2-normalize sahte embedding
            emb = np.random.randn(512).astype(np.float32)
            emb /= np.linalg.norm(emb)
            np.save(str(out_emb), emb)

            manifest_lines.append(f'{out_img}\t{out_emb}')

    manifest_path = str(out_path / 'manifest.txt')
    with open(manifest_path, 'w') as f:
        f.write('\n'.join(manifest_lines))
    print(f'Demo manifest: {len(manifest_lines)} örnek → {manifest_path}')

else:
    # Gerçek ön işleme
    manifest_path = preprocess_dataset(
        raw_dir     = cfg.data.data_dir,
        out_dir     = cfg.data.processed_dir,
        output_size = cfg.data.image_size,
        align_size  = getattr(cfg.data, 'align_size', 128),
        max_per_id  = 100,
        max_samples = getattr(cfg.data, 'max_samples', 100_000),
    )

print(f'Manifest: {manifest_path}')


In [ ]:
# Veri setini doğrula
from src.data.dataset import build_dataloaders
import torch

train_loader, val_loader, test_loader = build_dataloaders(
    manifest_path = manifest_path,
    image_size    = cfg.data.image_size,
    batch_size    = cfg.train.batch_size,
    val_split     = cfg.data.val_split,
    test_split    = cfg.data.test_split,
    num_workers   = cfg.data.num_workers,
)

print(f'Train: {len(train_loader.dataset):,} örnek  |  Val: {len(val_loader.dataset):,} örnek  |  Test: {len(test_loader.dataset):,} örnek')
print(f'Train batch sayısı: {len(train_loader)}')

# İlk batch'i incele
emb_batch, img_batch = next(iter(train_loader))
print(f'Embedding boyutu: {emb_batch.shape}   dtype: {emb_batch.dtype}')
print(f'Görüntü boyutu:   {img_batch.shape}   dtype: {img_batch.dtype}')
print(f'Görüntü değer aralığı: [{img_batch.min():.2f}, {img_batch.max():.2f}]')

In [ ]:
# Örnek görüntüleri göster
import matplotlib.pyplot as plt
from src.data.dataset import denormalize

imgs_show = denormalize(img_batch[:8]).permute(0, 2, 3, 1).cpu().numpy()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs_show[i].clip(0, 1))
    ax.axis('off')
plt.suptitle('Eğitim Görüntüleri (128×128)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Model Tanımı ve Parametre Raporu

In [ ]:
from src.models.decoder import FaceDecoder
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz: {device}')

model = FaceDecoder(
    embedding_dim    = cfg.model.embedding_dim,
    initial_spatial  = cfg.model.initial_spatial,
    initial_channels = cfg.model.initial_channels,
    decoder_channels = cfg.model.decoder_channels,
).to(device)

info = model.count_parameters()
print('\nFaceDecoder Parametre Raporu:')
print(f'  Toplam parametre : {info["total_params"]:,}')
print(f'  float32          : {info["float32_mb"]} MB')
print(f'  float16          : {info["float16_mb"]} MB')
print(f'  INT8 (TFLite)    : {info["int8_mb"]} MB')

# Hızlı forward test
dummy_z = torch.randn(4, 512).to(device)
dummy_out = model(dummy_z)
print(f'\nForward test: {dummy_z.shape} → {dummy_out.shape}  [{dummy_out.min():.3f}, {dummy_out.max():.3f}]')

## 7b. Faz A — 32 Örnek Overfit Sanity Check

Ana eğitime geçmeden önce, pipeline'ın öğrenebildiğini ucuza doğruluyoruz.
Final L1 < 0.03 ve görüntüler yakınsıyorsa pipeline sağlam; değilse ana
eğitime geçmeden bug'ı bul.

In [ ]:
from src.overfit_check import run_overfit_test

final_l1 = run_overfit_test(
    cfg            = cfg,
    manifest_path  = manifest_path,
    n_samples      = 32,
    steps          = 800,
    save_path      = '/content/overfit_check.png',
)


## 8. Eğitim

In [ ]:
from src.train import train

# Checkpoint'ten devam etmek için:
# resume_path = '/content/drive/MyDrive/face_recon/checkpoints/checkpoint_epoch0010.pt'
resume_path = None

best_ckpt = train(
    cfg           = cfg,
    manifest_path = manifest_path,
    resume_from   = resume_path,
)

print(f'\nEğitim tamamlandı. En iyi model: {best_ckpt}')

In [ ]:
# TensorBoard ile kayıpları izle
%load_ext tensorboard
%tensorboard --logdir {cfg.train.log_dir}

## 9. Sonuçları Görselleştir

In [ ]:
import torch
import matplotlib.pyplot as plt
from src.data.dataset import denormalize, build_val_transform
from src.models.decoder import FaceDecoder

# En iyi model yükle
state = torch.load(best_ckpt, map_location=device)
model.load_state_dict(state['model'])
model.eval()

# Val set'ten örnek al
val_embs, val_imgs = next(iter(val_loader))
val_embs = val_embs.to(device)

with torch.no_grad():
    generated = model(val_embs[:8])

# Görselleştir
real_np = denormalize(val_imgs[:8]).permute(0, 2, 3, 1).cpu().numpy()
gen_np  = denormalize(generated[:8]).permute(0, 2, 3, 1).cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i in range(8):
    axes[0, i].imshow(real_np[i].clip(0, 1))
    axes[0, i].set_title('Gerçek', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(gen_np[i].clip(0, 1))
    axes[1, i].set_title('Üretilen', fontsize=9)
    axes[1, i].axis('off')

plt.suptitle('Gerçek vs Üretilen Yüzler', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/face_recon/results_sample.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Identity Cosine Similarity hesapla
from facenet_pytorch import InceptionResnetV1
import torch.nn.functional as F

facenet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

all_scores = []
model.eval()

with torch.no_grad():
    for embs, real_imgs in val_loader:
        embs      = embs.to(device)
        real_imgs = real_imgs.to(device)
        generated = model(embs)

        # FaceNet için 160×160'a resize
        real_160 = F.interpolate(real_imgs,  size=(160, 160), mode='bilinear', align_corners=False)
        gen_160  = F.interpolate(generated,  size=(160, 160), mode='bilinear', align_corners=False)

        real_emb = F.normalize(facenet(real_160), dim=1)
        gen_emb  = F.normalize(facenet(gen_160),  dim=1)

        scores = F.cosine_similarity(real_emb, gen_emb).cpu().numpy()
        all_scores.extend(scores.tolist())

import numpy as np
all_scores = np.array(all_scores)
print(f'Val Identity Cosine Similarity:')
print(f'  Ortalama : {all_scores.mean():.4f}')
print(f'  Std      : {all_scores.std():.4f}')
print(f'  Medyan   : {np.median(all_scores):.4f}')
print(f'  > 0.5    : {(all_scores > 0.5).mean()*100:.1f}%  (hedef: yüksek)')

## 10. TFLite Export

In [ ]:
from src.export_tflite import export_model

exported_files = export_model(
    checkpoint_path = best_ckpt,
    export_dir      = cfg.export.export_dir,
    cfg             = cfg,
)

print('\nExport tamamlandı:')
for fmt, path in exported_files.items():
    import os
    size_mb = os.path.getsize(path) / 1e6
    print(f'  {fmt:12s}: {path}  ({size_mb:.2f} MB)')

In [ ]:
# TFLite modelini Colab'da test et
import tensorflow as tf
import numpy as np

tflite_path = exported_files.get('int8', exported_files.get('float32'))

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

in_detail  = interpreter.get_input_details()[0]
out_detail = interpreter.get_output_details()[0]

print(f'Giriş: {in_detail["shape"]}  dtype: {in_detail["dtype"]}')
print(f'Çıktı: {out_detail["shape"]}  dtype: {out_detail["dtype"]}')

# Test
test_emb = np.random.randn(1, 512).astype(np.float32)
test_emb /= np.linalg.norm(test_emb)  # L2 normalize

interpreter.set_tensor(in_detail['index'], test_emb)
interpreter.invoke()
output = interpreter.get_tensor(out_detail['index'])

print(f'\nTFLite çıktı boyutu: {output.shape}')
print(f'Değer aralığı: [{output.min():.3f}, {output.max():.3f}]')

# Görselleştir
import matplotlib.pyplot as plt
# PyTorch NCHW → NHWC veya doğrudan NHWC olabilir
if output.shape[1] == 3:  # NCHW
    img_vis = output[0].transpose(1, 2, 0)
else:  # NHWC
    img_vis = output[0]

img_vis = (img_vis * 0.5 + 0.5).clip(0, 1)
plt.figure(figsize=(4, 4))
plt.imshow(img_vis)
plt.title('TFLite Çıktısı')
plt.axis('off')
plt.show()

## 11. Android Entegrasyon Kodu

TFLite dosyasını Android projesine eklemek için:

1. `face_decoder_int8.tflite` dosyasını `app/src/main/assets/` klasörüne kopyalayın
2. `build.gradle` dosyasına ekleyin:
   ```gradle
   dependencies {
       implementation 'org.tensorflow:tensorflow-lite:2.14.0'
       implementation 'org.tensorflow:tensorflow-lite-gpu:2.14.0'
   }
   ```

In [ ]:
# Android Kotlin inference kodu
android_code = '''
// FaceReconstructionModel.kt
import android.content.Context
import android.graphics.Bitmap
import org.tensorflow.lite.Interpreter
import org.tensorflow.lite.gpu.GpuDelegate
import java.io.FileInputStream
import java.nio.ByteBuffer
import java.nio.ByteOrder
import java.nio.MappedByteBuffer
import java.nio.channels.FileChannel

class FaceReconstructionModel(context: Context) {

    private val interpreter: Interpreter
    private val GPU_DELEGATE = true
    private val EMBEDDING_DIM = 512
    private val IMAGE_SIZE = 128

    init {
        val options = Interpreter.Options().apply {
            if (GPU_DELEGATE) addDelegate(GpuDelegate())
            numThreads = 4
        }
        interpreter = Interpreter(loadModelFile(context), options)
    }

    private fun loadModelFile(context: Context): MappedByteBuffer {
        val fileDescriptor = context.assets.openFd("face_decoder_int8.tflite")
        val inputStream = FileInputStream(fileDescriptor.fileDescriptor)
        val fileChannel = inputStream.channel
        return fileChannel.map(
            FileChannel.MapMode.READ_ONLY,
            fileDescriptor.startOffset,
            fileDescriptor.declaredLength
        )
    }

    /**
     * QR koddan okunan embedding'i yüz görüntüsüne dönüştür.
     * @param embedding L2-normalized float array, boyut: [512]
     * @return 128×128 Bitmap
     */
    fun reconstruct(embedding: FloatArray): Bitmap {
        // Giriş buffer (float32, 1×512)
        val inputBuffer = ByteBuffer.allocateDirect(4 * EMBEDDING_DIM)
            .order(ByteOrder.nativeOrder())
        embedding.forEach { inputBuffer.putFloat(it) }
        inputBuffer.rewind()

        // Çıktı buffer (float32, 1×3×128×128 veya 1×128×128×3)
        val outputBuffer = ByteBuffer.allocateDirect(4 * 3 * IMAGE_SIZE * IMAGE_SIZE)
            .order(ByteOrder.nativeOrder())

        interpreter.run(inputBuffer, outputBuffer)
        outputBuffer.rewind()

        // [-1,1] → [0,255] dönüşümü ve Bitmap oluşturma
        return bufferToBitmap(outputBuffer)
    }

    private fun bufferToBitmap(buffer: ByteBuffer): Bitmap {
        val pixels = IntArray(IMAGE_SIZE * IMAGE_SIZE)
        // NCHW formatı: [R plane, G plane, B plane]
        val r = FloatArray(IMAGE_SIZE * IMAGE_SIZE)
        val g = FloatArray(IMAGE_SIZE * IMAGE_SIZE)
        val b = FloatArray(IMAGE_SIZE * IMAGE_SIZE)
        for (i in r.indices) r[i] = buffer.float
        for (i in g.indices) g[i] = buffer.float
        for (i in b.indices) b[i] = buffer.float

        for (i in pixels.indices) {
            val ri = ((r[i] * 0.5f + 0.5f) * 255f).toInt().coerceIn(0, 255)
            val gi = ((g[i] * 0.5f + 0.5f) * 255f).toInt().coerceIn(0, 255)
            val bi = ((b[i] * 0.5f + 0.5f) * 255f).toInt().coerceIn(0, 255)
            pixels[i] = (0xFF shl 24) or (ri shl 16) or (gi shl 8) or bi
        }

        return Bitmap.createBitmap(pixels, IMAGE_SIZE, IMAGE_SIZE, Bitmap.Config.ARGB_8888)
    }

    fun close() = interpreter.close()
}
'''

# Kodu Drive'a kaydet
android_path = '/content/drive/MyDrive/face_recon/export/FaceReconstructionModel.kt'
with open(android_path, 'w') as f:
    f.write(android_code)

print(f'Android Kotlin kodu kaydedildi: {android_path}')
print(android_code)

---
## Özet

| Adım | Durum |
|------|-------|
| Ön işleme (insightface) | Tamamlandı |
| Model eğitimi | Tamamlandı |
| TFLite INT8 export | Tamamlandı |
| Android Kotlin kodu | Tamamlandı |

**Sonraki adım:** `face_decoder_int8.tflite` ve `FaceReconstructionModel.kt` dosyalarını Android projesine entegre edin.